# Data generation

In [3]:
import requests

base_url = "https://inspirehep.net/api/literature"
target_categories = ["nucl-th", "hep-lat"]

print(f"{'Category':<15} | {'Total Papers (All-time)'}")
print("-" * 40)

for category in target_categories:
    try:
        params = {
            "q": f"primary_arxiv_category:{category}",
            "size": 1, 
        }
        
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        
        data = response.json()
        total_count = data.get("hits", {}).get("total", 0)
        
        print(f"{category:<15} | {total_count:,}")
        
    except Exception as e:
        print(f"{category:<15} | Error: {e}")


Category        | Total Papers (All-time)
----------------------------------------
nucl-th         | 35,146
hep-lat         | 18,823


In [ ]:
import time
import requests
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding
from tqdm.auto import tqdm 

COLLECTION_NAME = "physics_rag_collection_lat_nuc" 
VECTOR_SIZE = 768 # dimension of  BAAI/bge-base-en-v1.5
client = QdrantClient("http://localhost:6333")

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "default": models.VectorParams( # vector_name = 'default'
            size=VECTOR_SIZE,
            distance=models.Distance.COSINE
        )
    }
)
print(f"Collection '{COLLECTION_NAME}' created!")


embedding_model = TextEmbedding(model_name="BAAI/bge-base-en-v1.5", threads=4)

def get_vector(text):
    return next(embedding_model.embed([text])).tolist()

def ingest_data(target_count=10):
    base_url = "https://inspirehep.net/api/literature"
    page = 1
    count = 0
    
    pbar = tqdm(total=target_count)

    while count < target_count:
        params = {
            "q": "(primary_arxiv_category:nucl-th OR primary_arxiv_category:hep-lat)",
            "size": 500,
            "page": page,
            "sort": "mostrecent"
        }
        
        try:
            response = requests.get(base_url, params=params)
            if response.status_code == 400: # usually up to 10,000
                break
            response.raise_for_status()
            data = response.json()
            hits = data.get("hits", {}).get("hits", [])
            
            if not hits:
                break
                
            points_batch = []
            
            for hit in hits:
                metadata = hit.get("metadata", {})
                
                abstract = ""
                if "abstracts" in metadata and len(metadata["abstracts"]) > 0:
                    abstract = metadata["abstracts"][0].get("value", "")
            
                title = metadata.get("titles", [{}])[0].get("title", "")
                
                preprint_date = metadata.get("preprint_date")
                
                if not abstract:
                    continue
                    
     
                vector = get_vector(abstract)
                
            
                point = models.PointStruct(
                    id=int(hit.get("id")), 
                    vector={"default": vector}, 
                    payload={
                        "title": title,
                        "abstract": abstract,
                        "preprint_date": preprint_date,
                        "authors": len(metadata.get("authors", []))
                    }
                )
                points_batch.append(point)
                
            # batch upload
            if points_batch:
                client.upsert(
                    collection_name=COLLECTION_NAME,
                    points=points_batch
                )
                count += len(points_batch)
                pbar.update(len(points_batch))
            
            page += 1
            time.sleep(0.5)
            
        except Exception as e:
            print(f"Error at page {page}: {e}")
            break
            
    pbar.close()
    print("Ingestion Finished!")


ingest_data(target_count=10000) 

Collection 'physics_rag_collection_lat_nuc' created!


  0%|          | 0/10000 [00:00<?, ?it/s]

Ingestion Finished!


# RAG

In [18]:
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding

qdrant = QdrantClient("http://localhost:6333")
COLLECTION_NAME = "physics_rag_collection_lat_nuc"

embedding_model = TextEmbedding(model_name="BAAI/bge-base-en-v1.5", threads=4)

model_handle = "BAAI/bge-base-en-v1.5"


def get_query_vector(text):
    return next(embedding_model.embed([text])).tolist()


def search_papers(query, top_k=5):
    vector = get_query_vector(query)

    search_result = qdrant.query_points(collection_name=COLLECTION_NAME,
                                        query=vector,
                                        using="default",
                                        limit=top_k,
                                        with_payload=True
                                        )

    return search_result


results = search_papers("What is the signal-to-noise problem?")

for point in results.points:
        print(f"=== Score: {point.score:.4f} ===")
        print(f"Title: {point.payload.get('title')}")
        print(f"preprint date: {point.payload.get('preprint_date')}")
        
        print(f"Abstract: {point.payload.get('abstract', '')[:200]}...\n")


=== Score: 0.6640 ===
Title: Stochastic automatic differentiation and the signal to noise problem
preprint date: 2025-02-21
Abstract: Lattice Field theory allows to extract properties of particles in strongly coupled quantum field theories by studying Euclidean vacuum expectation values. When estimated from numerical Monte Carlo sim...

=== Score: 0.6210 ===
Title: Machine-learning approaches to accelerating lattice simulations
preprint date: 2025-02-04
Abstract: The last decade has seen an explosive growth of interest in exploiting developments in machine learning to accelerate lattice QCD calculations. On the sampling side, generative models are a promising ...

=== Score: 0.6151 ===
Title: Control variates for lattice field theory
preprint date: 2023-07-27
Abstract: In most lattice field theories, correlators are plagued by a signal-to-noise problem of exponential difficulty in the time separation. We propose a method for improving the signal-to-noise ratio, in w...

=== Score: 0.60

# Evaluate retrieval

In [2]:
import pandas as pd

df = pd.read_csv("../data/ground_truth_dataset.csv")

In [4]:
from qdrant_client import QdrantClient
from fastembed import TextEmbedding
from tqdm.auto import tqdm

COLLECTION_NAME = "physics_rag_collection_lat_nuc"
client = QdrantClient("http://localhost:6333")
embedding_model = TextEmbedding(model_name="BAAI/bge-base-en-v1.5", threads=4)


def get_query_vector(text):
    return next(embedding_model.embed([text])).tolist()


def evaluate_metrics(df, top_k=5):
    hits = 0
    mrr_sum = 0
    total = len(df)



    for _, row in tqdm(df.iterrows(), total=total):
        question = row['question']
        target_id = int(row['paper_id'])


        vector = get_query_vector(question)
        search_result = client.query_points(
            collection_name=COLLECTION_NAME,
            query=vector,
            using="default",
            limit=top_k,
            with_payload=False 
        )

        retrieved_ids = [point.id for point in search_result.points]

        # scoring
        if target_id in retrieved_ids:
            hits +=1

            rank = retrieved_ids.index(target_id) +1

            mrr_sum += (1.0/rank)
        
    hit_rate = hits / total if total > 0 else 0
    mrr = mrr_sum / total if total > 0 else 0
    
    return hit_rate, mrr


hit_rate, mmr = evaluate_metrics(df, top_k=5)
print(f"hit rate: {hit_rate}, MMR: {mmr}")

  0%|          | 0/997 [00:00<?, ?it/s]

hit rate: 0.9077231695085256, MMR: 0.8118522233366775


In [5]:
k_values = [1, 3, 5, 10, 20]
results = []


for k in k_values:
    hit_rate, mrr = evaluate_metrics(df, top_k=k)
    results.append({"Top_K": k, "Hit_Rate": hit_rate, "MRR": mrr})

res_df = pd.DataFrame(results)
res_df

  0%|          | 0/997 [00:00<?, ?it/s]

  0%|          | 0/997 [00:00<?, ?it/s]

  0%|          | 0/997 [00:00<?, ?it/s]

  0%|          | 0/997 [00:00<?, ?it/s]

  0%|          | 0/997 [00:00<?, ?it/s]

,Top_K,Hit_Rate,MRR
0,1,0.746239,0.746239
1,3,0.875627,0.804580
2,5,0.907723,0.811852
3,10,0.923771,0.814037
4,20,0.944835,0.815637


## Reranking

In [14]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def search_with_rerank(query, top_k_retrieval=50, top_k_final=5):
    vector = get_query_vector(query)
    search_result = client.query_points(
        collection_name=COLLECTION_NAME,
        query=vector,
        using="default",
        limit=top_k_retrieval,
        with_payload=True 
    )
    
    passages = []

    for hit in search_result.points:
        passages.append([query, hit.payload['title'] + ": " + hit.payload['abstract']])

    scores = reranker.predict(passages)

    reranked_results = []
    for hit, score in zip(search_result.points, scores):
        reranked_results.append({
            "id": hit.id,
            "score": score,
            "payload": hit.payload,
            "original_score": hit.score
        })
    reranked_results = sorted(reranked_results, key=lambda x: x['score'], reverse=True)
    
    return reranked_results[:top_k_final]

results = search_with_rerank("What is the signal-to-noise problem?")

for res in results:
    print(f"Score: {res['score']:.4f} | Title: {res['payload']['title']}")

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Score: 7.2326 | Title: Stochastic automatic differentiation and the signal to noise problem
Score: 5.7170 | Title: Control variates for lattice field theory
Score: 4.6938 | Title: Machine-learning approaches to accelerating lattice simulations
Score: 3.6398 | Title: On the equivalence of Prony and Lanczos methods for Euclidean correlation functions
Score: 3.2981 | Title: Lanczos algorithm for lattice QCD matrix elements


In [15]:
results

[{'id': 2893257,
  'score': np.float32(7.2326493),
  'payload': {'title': 'Stochastic automatic differentiation and the signal to noise problem',
   'abstract': 'Lattice Field theory allows to extract properties of particles in strongly coupled quantum field theories by studying Euclidean vacuum expectation values. When estimated from numerical Monte Carlo simulations these are typically affected by the so called Signal to Noise problem: both the signal and the variance decay exponentially with the Euclidean time, but the variance decays slower, making the signal to noise ratio to degrade exponentially fast. In this work we show that writing correlators as derivatives with respect to sources and evaluating these derivatives using techniques of stochastic automatic differentiation can eliminate completely the signal to noise problem. We show some results in scalar field theories, and comment on the prospects for applicability in Gauge theories and QCD.',
   'preprint_date': '2025-02-21'

In [16]:
def evaluate_metrics_rerank(df, top_k_retrieval=50, top_k_final=5):
    hits = 0
    mrr_sum = 0
    total = len(df)



    for _, row in tqdm(df.iterrows(), total=total):
        question = row['question']
        target_id = int(row['paper_id'])


        search_result = search_with_rerank(question, top_k_retrieval, top_k_final)


        retrieved_ids = [point['id'] for point in search_result]

        # scoring
        if target_id in retrieved_ids:
            hits +=1

            rank = retrieved_ids.index(target_id) +1

            mrr_sum += (1.0/rank)
        
    hit_rate = hits / total if total > 0 else 0
    mrr = mrr_sum / total if total > 0 else 0
    
    return hit_rate, mrr


hit_rate, mmr = evaluate_metrics_rerank(df, top_k_retrieval=50, top_k_final=5)
print(f"hit rate: {hit_rate}, MMR: {mmr}")

  0%|          | 0/997 [00:00<?, ?it/s]

hit rate: 0.8976930792377131, MMR: 0.8343363423604149


While it has better performance in MMR, it takes more than 10 times longer and uses too much resources. Let us go with `top_k=5` for dense search.

# RAG Evaluation (LLM as a judge)

In [39]:
import pandas as pd
from qdrant_client import QdrantClient
from fastembed import TextEmbedding
from google import genai
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tqdm.auto import tqdm



df = pd.read_csv("../data/ground_truth_dataset.csv")

COLLECTION_NAME = "physics_rag_collection_lat_nuc"
client = QdrantClient("http://localhost:6333")
embedding_model = TextEmbedding(model_name="BAAI/bge-base-en-v1.5", threads=4)

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=api_key)


def get_query_vector(text):
    return next(embedding_model.embed([text])).tolist()


prompt_template = """
You are an expert theoretical physicist assisting a junior researcher. 
Use the following pieces of retrieved context to answer the question. 
If the answer is not in the context, just say that you don't know based on the provided documents.

Question: {query}

Context (Retrieved Papers): {context_text}

""".strip()


def retrieve_context(query, top_k=5):
    query_vector = get_query_vector(query)

    search_result = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        using="default",
        limit=top_k,
        with_payload=True
    )

    # Formatting context
    contexts = []
    for hit in search_result.points:
        title = hit.payload['title']
        preprint_date = hit.payload['preprint_date']
        abstract = hit.payload['abstract']

        # Title: ... (Year: ...)
        # Abstract: ...
        formatted_text = f"Title: {title} ({preprint_date})\nAbstract: {abstract}"
        contexts.append(formatted_text)

    return "\n\n---\n\n".join(contexts)


class RAGOutput(BaseModel):
    answer: str=Field(..., description = "Answer for the question based on context (It can include LaTeX equations)")


def rag(query, model="models/gemini-2.5-flash-lite"):
    context = retrieve_context(query)
    prompt = prompt_template.format(query=query, context_text=context)

    response = gemini_client.models.generate_content(
        model=model,
        contents=prompt,
        config={
                "response_schema": RAGOutput,
            }
    )

    return response.text, context


rag("What is control variates?")

('Control variates are a method used to reduce the uncertainty (variance) in results obtained from stochastic methods, such as Monte Carlo (MC) simulations. The core idea is to compute the expectation value of the difference between the observable of interest and another observable. This auxiliary observable should be correlated with the observable of interest, and its average should be known (often zero). By using this correlated auxiliary observable, the variance of the estimator for the main observable can be significantly reduced while maintaining unbiasedness.\n\nTraditionally, control variates were often constructed using heuristic approaches or educated guesses, which might not be optimal for complex theories. More recently, neural networks have been utilized to parametrize these control variates, eliminating the need for manual guesswork and allowing for more effective variance reduction, especially in challenging regimes like the strong coupling regime or high-dimensional prob

In [41]:
answers = []
retrieved_contexts = []

for q in tqdm(df['question']):
    answer, context = rag(q)
    answers.append(answer)
    retrieved_contexts.append(context)

df['answer'] = answers
df['contexts'] = retrieved_contexts 

df.to_json("../data/rag_results_for_eval_gemini-2.5-flash-lite.json", orient="records")

  0%|          | 0/997 [00:00<?, ?it/s]

In [42]:
import json

prompt_relevance_template = """
You are an expert evaluator for a RAG system.
Your task is to analyze the relevance of the generated answer to the given question.
Based on the relevance of the generated answer, you will classify it
as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Here is the data for evaluation:

Question: {question}
Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON without using code blocks:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()



def evaluate_relevance(question, answer):

    prompt = prompt_relevance_template.format(
        question=question, answer_llm=answer)
    try:
        response = gemini_client.models.generate_content(
            model="models/gemini-2.5-flash-lite",
            contents=prompt,
            config={
                "response_mime_type": "application/json",
                "max_output_tokens": 1000
            }
        )

        return json.loads(response.text)
    except (json.JSONDecodeError, ValueError, Exception) as e:
        print(f"Error: {e}")
        return {"Relevance": None, "Explanation": None}


eval_relevance = []
eval_explanation = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    q = row['question']
    a = row['answer']

    result = evaluate_relevance(q, a)

    eval_relevance.append(result.get("Relevance"))
    eval_explanation.append(result.get("Explanation"))

df["relevance"] = eval_relevance
df["explanation"] = eval_explanation


  0%|          | 0/997 [00:00<?, ?it/s]

Error: Invalid \escape: line 3 column 173 (char 201)
Error: Invalid \escape: line 3 column 128 (char 156)
Error: Invalid \escape: line 3 column 205 (char 233)
Error: Invalid \escape: line 3 column 237 (char 265)
Error: Invalid \escape: line 3 column 220 (char 248)
Error: Invalid \escape: line 3 column 279 (char 314)
Error: Invalid \escape: line 3 column 133 (char 161)
Error: Invalid \escape: line 3 column 173 (char 201)
Error: Invalid \escape: line 3 column 117 (char 145)
Error: Invalid \escape: line 3 column 131 (char 159)
Error: Invalid \escape: line 3 column 99 (char 127)
Error: Invalid \escape: line 3 column 171 (char 199)
Error: Invalid \escape: line 3 column 250 (char 278)
Error: Expecting ',' delimiter: line 3 column 318 (char 346)
Error: Invalid \escape: line 3 column 52 (char 87)
Error: Invalid \escape: line 3 column 130 (char 158)
Error: Invalid \escape: line 3 column 174 (char 202)
Error: Invalid \escape: line 3 column 107 (char 135)
Error: Invalid \escape: line 3 column 84 

In [48]:
df.relevance.value_counts(normalize=True)

relevance
RELEVANT           0.804560
PARTLY_RELEVANT    0.134636
NON_RELEVANT       0.060803
Name: proportion, dtype: float64

In [49]:
prompt_faithfulness_template = """
You are an expert evaluator for a RAG system.
Your task is to verify if the generated answer is grounded in the retrieved context.
If the answer contains information NOT present in the context, it is a Hallucination.

Here is the data:

Retrieved Context: {context}

Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON without using code blocks:

Evaluate the faithfulness:
- "FAITHFUL": All claims in the answer are supported by the context.
- "HALLUCINATED": The answer contains information not found in the context.

Provide your evaluation in JSON:
{{
  "Faithfulness": "FAITHFUL" | "HALLUCINATED",
  "Reason": "[Which part is hallucinated?]"
}}
""".strip()


def evaluate_faithfulness(context, answer):
    prompt = prompt_faithfulness_template.format(context=context, answer_llm=answer)

    try:
        response = gemini_client.models.generate_content(
            model="models/gemini-2.5-flash-lite",
            contents=prompt,
            config={
                "response_mime_type": "application/json",
                "max_output_tokens": 1000
            }
        )

        return json.loads(response.text)
    except (json.JSONDecodeError, ValueError, Exception) as e:
        print(f"Error: {e}")
        return {"Faithfulness": None, "Reason": None}

faithfulnesses = []
reasons = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    c = row['contexts']
    a = row['answer']

    result = evaluate_faithfulness(c, a)

    faithfulnesses.append(result.get("Faithfulness"))
    reasons.append(result.get("Reason"))

df["faithfulness"] = faithfulnesses
df["reason"] = reasons

  0%|          | 0/997 [00:00<?, ?it/s]

Error: Invalid \escape: line 3 column 205 (char 236)
Error: Expecting ',' delimiter: line 3 column 203 (char 234)
Error: Invalid \escape: line 3 column 122 (char 153)
Error: Invalid \escape: line 3 column 152 (char 183)
Error: Invalid \escape: line 3 column 177 (char 208)
Error: Invalid \escape: line 3 column 177 (char 208)
Error: Invalid \escape: line 3 column 169 (char 200)
Error: Invalid \escape: line 3 column 141 (char 172)
Error: Invalid \escape: line 3 column 347 (char 378)
Error: Invalid \escape: line 3 column 56 (char 87)
Error: Invalid \escape: line 3 column 222 (char 253)
Error: Invalid \escape: line 3 column 171 (char 202)
Error: Invalid \escape: line 3 column 51 (char 82)
Error: Expecting ',' delimiter: line 3 column 111 (char 142)
Error: Expecting ',' delimiter: line 3 column 551 (char 582)
Error: Invalid \escape: line 3 column 384 (char 415)
Error: Invalid \escape: line 3 column 159 (char 190)
Error: Invalid \escape: line 3 column 225 (char 256)
Error: Invalid \escape: li

In [50]:
df.faithfulness.value_counts(normalize=True)

faithfulness
FAITHFUL        0.968815
HALLUCINATED    0.031185
Name: proportion, dtype: float64

In [51]:
df.to_json("../data/rag_results_for_eval_gemini-2.5-flash-lite.json", orient="records")

## gemini-2.5-flash

In [52]:
df = pd.read_csv("../data/ground_truth_dataset.csv")

COLLECTION_NAME = "physics_rag_collection_lat_nuc"
client = QdrantClient("http://localhost:6333")
embedding_model = TextEmbedding(model_name="BAAI/bge-base-en-v1.5", threads=4)

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=api_key)

In [53]:
answers = []
retrieved_contexts = []

for q in tqdm(df['question']):
    answer, context = rag(q, model="models/gemini-2.5-flash")
    answers.append(answer)
    retrieved_contexts.append(context)

df['answer'] = answers
df['contexts'] = retrieved_contexts 

df.to_json("../data/rag_results_for_eval_gemini-2.5-flash.json", orient="records")

  0%|          | 0/997 [00:00<?, ?it/s]

In [54]:
eval_relevance = []
eval_explanation = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    q = row['question']
    a = row['answer']

    result = evaluate_relevance(q, a)

    eval_relevance.append(result.get("Relevance"))
    eval_explanation.append(result.get("Explanation"))

df["relevance"] = eval_relevance
df["explanation"] = eval_explanation

df.relevance.value_counts(normalize=True)

  0%|          | 0/997 [00:00<?, ?it/s]

Error: Invalid \escape: line 3 column 458 (char 486)
Error: Invalid \escape: line 3 column 114 (char 142)
Error: Invalid \escape: line 3 column 124 (char 152)
Error: Invalid \escape: line 3 column 121 (char 149)
Error: Invalid \escape: line 3 column 111 (char 139)
Error: Invalid \escape: line 3 column 167 (char 199)
Error: Invalid \escape: line 3 column 173 (char 201)
Error: Invalid \escape: line 3 column 116 (char 144)
Error: Invalid \escape: line 3 column 107 (char 135)
Error: Invalid \escape: line 3 column 121 (char 149)
Error: Invalid \escape: line 3 column 118 (char 153)
Error: Invalid \escape: line 3 column 151 (char 179)
Error: Invalid \escape: line 3 column 144 (char 172)
Error: Invalid \escape: line 3 column 204 (char 232)
Error: Invalid \escape: line 3 column 129 (char 157)
Error: Invalid \escape: line 3 column 130 (char 158)
Error: Invalid \escape: line 3 column 176 (char 204)
Error: Invalid \escape: line 3 column 116 (char 144)
Error: Invalid \escape: line 3 column 152 (cha

relevance
RELEVANT           0.819936
PARTLY_RELEVANT    0.120043
NON_RELEVANT       0.060021
Name: proportion, dtype: float64

In [55]:
faithfulnesses = []
reasons = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    c = row['contexts']
    a = row['answer']

    result = evaluate_faithfulness(c, a)

    faithfulnesses.append(result.get("Faithfulness"))
    reasons.append(result.get("Reason"))

df["faithfulness"] = faithfulnesses
df["reason"] = reasons

df.faithfulness.value_counts(normalize=True)

  0%|          | 0/997 [00:00<?, ?it/s]

Error: Expecting ',' delimiter: line 3 column 148 (char 179)
Error: Invalid \escape: line 3 column 111 (char 142)
Error: Invalid \escape: line 3 column 93 (char 124)
Error: Invalid \escape: line 3 column 287 (char 318)
Error: Invalid \escape: line 3 column 41 (char 72)
Error: Invalid \escape: line 3 column 129 (char 160)
Error: Invalid \escape: line 3 column 207 (char 238)
Error: Invalid \escape: line 3 column 515 (char 550)
Error: Invalid \escape: line 3 column 160 (char 191)
Error: Invalid \escape: line 3 column 157 (char 192)
Error: Invalid \escape: line 3 column 158 (char 189)
Error: Invalid \escape: line 3 column 173 (char 204)
Error: Invalid \escape: line 3 column 263 (char 294)
Error: Expecting ',' delimiter: line 3 column 740 (char 775)
Error: Invalid \escape: line 3 column 146 (char 177)
Error: Invalid \escape: line 3 column 306 (char 337)
Error: Invalid \escape: line 3 column 194 (char 225)
Error: Invalid \escape: line 3 column 142 (char 173)
Error: Invalid \escape: line 3 co

faithfulness
FAITHFUL        0.980372
HALLUCINATED    0.019628
Name: proportion, dtype: float64

In [56]:
df.to_json("../data/rag_results_for_eval_gemini-2.5-flash.json", orient="records")

While `flash` model has a slightly better performance, considering its cost and time (~6 times), it is better to use `flash-lite` model.